# La part des profits dans le gâteau · *Profits' slice of the pie*

Notebook compagnon du chapitre **9. Comprendre le PIB : la mesure de la richesse produite par un pays** — [lire l'article](https://nmlab.io/ressources/comprendre-le-pib).
Companion notebook to chapter **9. Understanding GDP: The Measure of the Wealth a Country Produces** — [read the article](https://nmlab.io/en/ressources/understanding-gdp).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_profit_share() -> Series:
    """Profits des sociétés américaines après impôt (CP) en % du PIB (GDP), en direct de FRED.
    U.S. after-tax corporate profits as a share of GDP, live from FRED."""
    profits = nm.load_fred("CP")
    gdp = nm.load_fred("GDP")
    return (profits / gdp * 100).dropna()

share = load_profit_share()


import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="La part des profits dans le gâteau",
        sub="profits des sociétés américaines après impôt, en % du PIB",
        ylab="% du PIB",
        note="Sources : BEA via FRED (CP, GDP) — millésime du 15 juillet 2026.\n"
             "Les profits suivent le PIB nominal de loin : leur part oscille du simple au triple selon les décennies."),
    "en": dict(
        title="Profits' slice of the pie",
        sub="U.S. after-tax corporate profits, as a share of GDP",
        ylab="% of GDP",
        note="Sources: BEA via FRED (CP, GDP) — vintage of July 15, 2026.\n"
             "Profits track nominal GDP only loosely: their share swings threefold across decades."),
}

def build_figure(share: Series, lang: str) -> Figure:
    """Part des profits dans le PIB : ligne bleue, moyenne en pointillés, creux et pic annotés."""
    text = LABELS[lang]
    mean = share.mean()
    min_date, min_val = share.idxmin(), share.min()
    last_date, last_val = share.index[-1], share.iloc[-1]
    last_q = (last_date.month - 1) // 3 + 1
    if lang == "fr":
        mean_lbl = f"moyenne {share.index[0].year}-{last_date.year} : {mean:.1f} %".replace(".", ",")
        min_lbl = f"{min_date.year} : {min_val:.1f} %".replace(".", ",")
        prefix = "début" if last_q <= 2 else "fin"
        last_lbl = f"{prefix} {last_date.year} : {last_val:.1f} %".replace(".", ",")
    else:
        mean_lbl = f"{share.index[0].year}-{last_date.year} average: {mean:.1f}%"
        min_lbl = f"{min_date.year}: {min_val:.1f}%"
        prefix = "early" if last_q <= 2 else "late"
        last_lbl = f"{prefix} {last_date.year}: {last_val:.1f}%"

    fig = nm.figure(height_px=1026)
    ax = nm.axes(fig, left=0.082)
    ax.axhline(mean, color=nm.COLORS["muted"], linewidth=1.8, linestyle=(0, (7, 6)))
    ax.plot(share.index, share, color=nm.COLORS["blue"], linewidth=2.6)
    ax.set_ylim(2, 14)
    ax.set_yticks(range(2, 15, 2))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.text(share.index[-1], mean + 0.25, mean_lbl, ha="right", va="bottom",
            fontsize=21, color=nm.COLORS["muted"])
    ax.annotate(min_lbl, xy=(min_date, min_val), xytext=(70, -46),
                textcoords="offset points", ha="left", va="center", fontsize=22,
                color=nm.COLORS["text"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    ax.annotate(last_lbl, xy=(last_date, last_val), xytext=(-6, 72),
                textcoords="offset points", ha="right", va="center", fontsize=22,
                color=nm.COLORS["text"],
                arrowprops=dict(arrowstyle="-", color=nm.COLORS["muted"], lw=1.3))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(share, LANG)